In [52]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from config import DATABASE_PATH

con = duckdb.connect(str(DATABASE_PATH))

df = con.execute(
    """
    SELECT
        c.*,

        o.DS_CARGO,

        p.SG_PARTIDO,
        p.NM_PARTIDO,

        l.SG_UF,
        l.SG_UE,
        l.NM_UE,

        e.ANO_ELEICAO,
        e.NR_TURNO,
        e.DS_ELEICAO,
        e.DT_ELEICAO

    FROM dim_candidacy c

    LEFT JOIN dim_office o
        ON c.CD_CARGO = o.CD_CARGO

    LEFT JOIN dim_party p
        ON c.NR_PARTIDO = p.NR_PARTIDO

    LEFT JOIN dim_location l
        ON c.LOCATION_KEY = l.LOCATION_KEY

    LEFT JOIN dim_election e
        ON c.CD_ELEICAO = e.CD_ELEICAO
    """
).fetchdf()

con.close()

df.head()

,CANDIDACY_KEY,CD_ELEICAO,SQ_CANDIDATO,NR_PARTIDO,CD_CARGO,LOCATION_KEY,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,...,DS_CARGO,SG_PARTIDO,NM_PARTIDO,SG_UF,SG_UE,NM_UE,ANO_ELEICAO,NR_TURNO,DS_ELEICAO,DT_ELEICAO
0,6257|280002539826,6257,280002539826,30,1,FEDERAL|BR,30,ROMEU ZEMA NETO,ZEMA,NaN,...,PRESIDENTE,NOVO,PARTIDO NOVO,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04
1,6257|280002540693,6257,280002540693,14,2,FEDERAL|BR,14,AROLDO MEDINA,CORONEL MEDINA,NaN,...,VICE-PRESIDENTE,MISSÃO,PARTIDO MISSÃO,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04
2,6257|280002541458,6257,280002541458,16,2,FEDERAL|BR,16,VANESSA PORTUGAL BARBOSA,VANESSA PORTUGAL,NaN,...,VICE-PRESIDENTE,PSTU,PARTIDO SOCIALISTA DOS TRABALHADORES UNIFICADO,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04
3,6257|280002548140,6257,280002548140,35,2,FEDERAL|BR,35,SUÊD HAIDAR NOGUEIRA,SUÊD HAIDAR,NaN,...,VICE-PRESIDENTE,DEMOCRATA,DEMOCRATA,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04
4,6257|280002551543,6257,280002551543,22,2,FEDERAL|BR,22,ALFREDO GASPAR DE MENDONÇA NETO,ALFREDO GASPAR,NaN,...,VICE-PRESIDENTE,PL,PARTIDO LIBERAL,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04


In [53]:
# Visão geral

print(f"Candidaturas: {len(df):,}")
print(f"Partidos: {df['NR_PARTIDO'].nunique()}")
print(f"Cargos: {df['CD_CARGO'].nunique()}")
print(f"UFs / unidades: {df['LOCATION_KEY'].nunique()}")

Candidaturas: 20,530
Partidos: 30
Cargos: 10
UFs / unidades: 28


In [54]:
df["IDADE_NA_ELEICAO"].describe()

count    20530.000000
mean        49.205894
std         11.714488
min         19.000000
25%         41.000000
50%         49.000000
75%         57.000000
max         92.000000
Name: IDADE_NA_ELEICAO, dtype: float64

In [55]:
bins = [
    18,
    29,
    39,
    49,
    59,
    69,
    120,
]

labels = [
    "18–29",
    "30–39",
    "40–49",
    "50–59",
    "60–69",
    "70+",
]

df["FAIXA_ETARIA"] = pd.cut(
    df["IDADE_NA_ELEICAO"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

In [56]:
# Candidaturas por faixa etária
df["FAIXA_ETARIA"].value_counts()

FAIXA_ETARIA
40–49    6563
50–59    5684
30–39    3273
60–69    3119
18–29     952
70+       939
Name: count, dtype: int64

In [57]:
# Cargos
df["DS_CARGO"].value_counts()

DS_CARGO
DEPUTADO ESTADUAL     11103
DEPUTADO FEDERAL       7636
DEPUTADO DISTRITAL      420
2º SUPLENTE             319
1º SUPLENTE             318
SENADOR                 315
VICE-GOVERNADOR         197
GOVERNADOR              196
PRESIDENTE               13
VICE-PRESIDENTE          13
Name: count, dtype: int64

In [58]:
# Candidaturas por gênero
df["DS_GENERO"].value_counts()

DS_GENERO
MASCULINO    13374
FEMININO      7156
Name: count, dtype: int64

In [59]:
# Escolaridade
df["DS_GRAU_INSTRUCAO"].value_counts()

DS_GRAU_INSTRUCAO
SUPERIOR COMPLETO                12025
ENSINO MÉDIO COMPLETO             4395
SUPERIOR INCOMPLETO               2172
ENSINO MÉDIO INCOMPLETO            731
ENSINO FUNDAMENTAL INCOMPLETO      604
ENSINO FUNDAMENTAL COMPLETO        530
LÊ E ESCREVE                        73
Name: count, dtype: int64

In [60]:
# Cor/raça
df["DS_COR_RACA"].value_counts()

DS_COR_RACA
BRANCA      10013
PARDA        7418
PRETA        2800
INDÍGENA      191
AMARELA       108
Name: count, dtype: int64

In [61]:
# Ocupações
df["DS_OCUPACAO"].value_counts().head(20)

DS_OCUPACAO
OUTROS                                            2709
EMPRESÁRIO                                        2576
ADVOGADO                                          1691
DEPUTADO                                           874
VEREADOR                                           811
POLICIAL MILITAR                                   569
MÉDICO                                             558
SERVIDOR PÚBLICO ESTADUAL                          541
COMERCIANTE                                        532
ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS     523
ADMINISTRADOR                                      520
APOSENTADO (EXCETO SERVIDOR PÚBLICO)               464
SERVIDOR PÚBLICO MUNICIPAL                         435
PROFESSOR DE ENSINO MÉDIO                          382
JORNALISTA E REDATOR                               349
PROFESSOR DE ENSINO SUPERIOR                       289
ENGENHEIRO                                         289
PROFESSOR DE ENSINO FUNDAMENTAL                    28

In [62]:
# Partido
df["NR_PARTIDO"].value_counts()


NR_PARTIDO
22    1628
15    1391
30    1296
10    1296
20    1216
55    1117
12    1099
13    1095
45    1066
40     958
70     957
44     828
50     787
27     754
11     709
33     569
14     541
77     511
25     485
36     457
35     403
18     350
43     232
80     188
29     146
65     146
23     146
16     126
21      23
28      10
Name: count, dtype: int64

In [63]:
# Nome social
df["TEM_NOME_SOCIAL"].value_counts()


TEM_NOME_SOCIAL
False    20477
True        53
Name: count, dtype: int64

In [64]:

# Federação
df["TEM_FEDERACAO"].value_counts()

TEM_FEDERACAO
False    14175
True      6355
Name: count, dtype: int64

In [65]:
(
    df.groupby("DS_OCUPACAO")
    .size()
    .sort_values(ascending=False)
    .head(20)
)

DS_OCUPACAO
OUTROS                                            2709
EMPRESÁRIO                                        2576
ADVOGADO                                          1691
DEPUTADO                                           874
VEREADOR                                           811
POLICIAL MILITAR                                   569
MÉDICO                                             558
SERVIDOR PÚBLICO ESTADUAL                          541
COMERCIANTE                                        532
ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS     523
ADMINISTRADOR                                      520
APOSENTADO (EXCETO SERVIDOR PÚBLICO)               464
SERVIDOR PÚBLICO MUNICIPAL                         435
PROFESSOR DE ENSINO MÉDIO                          382
JORNALISTA E REDATOR                               349
ENGENHEIRO                                         289
PROFESSOR DE ENSINO SUPERIOR                       289
PROFESSOR DE ENSINO FUNDAMENTAL                    28

In [66]:
# Idade média por cargo
(
    df.groupby("DS_CARGO")["IDADE_NA_ELEICAO"]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_values("mean", ascending=False)
)

,count,mean,median,min,max
DS_CARGO,,,,,
VICE-PRESIDENTE,13,58.000000,59.0,34,73
PRESIDENTE,13,57.384615,56.0,39,80
SENADOR,315,55.444444,56.0,35,92
2º SUPLENTE,319,53.761755,52.0,31,86
1º SUPLENTE,318,53.503145,52.0,33,89
VICE-GOVERNADOR,197,51.228426,50.0,30,82
GOVERNADOR,196,50.688776,50.0,30,74
DEPUTADO ESTADUAL,11103,49.068630,49.0,19,92
DEPUTADO FEDERAL,7636,48.730225,48.0,19,89


In [67]:
# Escolaridade por cargo
pd.crosstab(
    df["DS_CARGO"],
    df["DS_GRAU_INSTRUCAO"],
    normalize="index"
)

DS_GRAU_INSTRUCAO,ENSINO FUNDAMENTAL COMPLETO,ENSINO FUNDAMENTAL INCOMPLETO,ENSINO MÉDIO COMPLETO,ENSINO MÉDIO INCOMPLETO,LÊ E ESCREVE,SUPERIOR COMPLETO,SUPERIOR INCOMPLETO
DS_CARGO,,,,,,,
1º SUPLENTE,0.025157,0.018868,0.128931,0.028302,0.009434,0.735849,0.053459
2º SUPLENTE,0.034483,0.025078,0.200627,0.021944,0.000000,0.639498,0.078370
DEPUTADO DISTRITAL,0.011905,0.026190,0.154762,0.021429,0.000000,0.685714,0.100000
DEPUTADO ESTADUAL,0.026569,0.032153,0.240295,0.037918,0.003603,0.554535,0.104927
DEPUTADO FEDERAL,0.026585,0.028156,0.193164,0.035752,0.003667,0.598219,0.114458
GOVERNADOR,0.015306,0.010204,0.102041,0.015306,0.000000,0.785714,0.071429
PRESIDENTE,0.076923,0.000000,0.000000,0.000000,0.000000,0.846154,0.076923
SENADOR,0.006349,0.003175,0.111111,0.015873,0.003175,0.793651,0.066667
VICE-GOVERNADOR,0.010152,0.020305,0.131980,0.020305,0.005076,0.751269,0.060914


In [68]:
# Gênero por cargo
pd.crosstab(
    df["DS_CARGO"],
    df["DS_GENERO"],
    normalize="index"
)

DS_GENERO,FEMININO,MASCULINO
DS_CARGO,,
1º SUPLENTE,0.308176,0.691824
2º SUPLENTE,0.307210,0.692790
DEPUTADO DISTRITAL,0.354762,0.645238
DEPUTADO ESTADUAL,0.344592,0.655408
DEPUTADO FEDERAL,0.365636,0.634364
GOVERNADOR,0.168367,0.831633
PRESIDENTE,0.153846,0.846154
SENADOR,0.219048,0.780952
VICE-GOVERNADOR,0.426396,0.573604


In [69]:
# Raça por cargo
pd.crosstab(
    df["DS_CARGO"],
    df["DS_COR_RACA"],
    normalize="index"
)

DS_COR_RACA,AMARELA,BRANCA,INDÍGENA,PARDA,PRETA
DS_CARGO,,,,,
1º SUPLENTE,0.003145,0.581761,0.009434,0.301887,0.103774
2º SUPLENTE,0.000000,0.460815,0.006270,0.407524,0.125392
DEPUTADO DISTRITAL,0.009524,0.397619,0.007143,0.442857,0.142857
DEPUTADO ESTADUAL,0.004954,0.474466,0.008917,0.374133,0.137530
DEPUTADO FEDERAL,0.005631,0.500262,0.009822,0.347695,0.136590
GOVERNADOR,0.010204,0.571429,0.015306,0.285714,0.117347
PRESIDENTE,0.000000,0.769231,0.000000,0.076923,0.153846
SENADOR,0.006349,0.609524,0.012698,0.279365,0.092063
VICE-GOVERNADOR,0.005076,0.527919,0.010152,0.248731,0.208122


In [70]:
age_group = df["FAIXA_ETARIA"]
labels = [
    "18–29",
    "30–39",
    "40–49",
    "50–59",
    "60–69",
    "70+",
]


condlist = [
    age_group == labels[0],   # Condition 1
    age_group == labels[1],
    age_group == labels[2],
    age_group == labels[3],
    age_group == labels[4],
    age_group == labels[5]
]

# Define corresponding outcomes
choicelist = [
    1,      # Action for condition 1: negate the value
    2,
    3,
    4,
    5,
    6      # Action for condition 2: square the value
]

# Run selection with a default value of 42 for remaining elements (where x == 3)
df["IND_FAIXA_ETARIA"] = np.select(condlist, choicelist, default=6)
df

,CANDIDACY_KEY,CD_ELEICAO,SQ_CANDIDATO,NR_PARTIDO,CD_CARGO,LOCATION_KEY,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,...,NM_PARTIDO,SG_UF,SG_UE,NM_UE,ANO_ELEICAO,NR_TURNO,DS_ELEICAO,DT_ELEICAO,FAIXA_ETARIA,IND_FAIXA_ETARIA
0,6257|280002539826,6257,280002539826,30,1,FEDERAL|BR,30,ROMEU ZEMA NETO,ZEMA,NaN,...,PARTIDO NOVO,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04,60–69,5
1,6257|280002540693,6257,280002540693,14,2,FEDERAL|BR,14,AROLDO MEDINA,CORONEL MEDINA,NaN,...,PARTIDO MISSÃO,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04,60–69,5
2,6257|280002541458,6257,280002541458,16,2,FEDERAL|BR,16,VANESSA PORTUGAL BARBOSA,VANESSA PORTUGAL,NaN,...,PARTIDO SOCIALISTA DOS TRABALHADORES UNIFICADO,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04,50–59,4
3,6257|280002548140,6257,280002548140,35,2,FEDERAL|BR,35,SUÊD HAIDAR NOGUEIRA,SUÊD HAIDAR,NaN,...,DEMOCRATA,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04,60–69,5
4,6257|280002551543,6257,280002551543,22,2,FEDERAL|BR,22,ALFREDO GASPAR DE MENDONÇA NETO,ALFREDO GASPAR,NaN,...,PARTIDO LIBERAL,BR,BR,BRASIL,2026,1,Eleição Geral Federal 2026,2026-10-04,50–59,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20525,6259|270002550608,6259,270002550608,25,7,ESTADUAL|TO,25444,ROSIMARIA DE SOUZA SANTOS,ROSE,NaN,...,PARTIDO RENOVACAO DEMOCRATICA,TO,TO,TOCANTINS,2026,1,Eleições Gerais Estaduais 2026,2026-10-04,30–39,2
20526,6259|270002550616,6259,270002550616,30,7,ESTADUAL|TO,30010,ADRIANO ALVES DE SOUZA,ADRIANO ROYAL PET,NaN,...,PARTIDO NOVO,TO,TO,TOCANTINS,2026,1,Eleições Gerais Estaduais 2026,2026-10-04,30–39,2
20527,6259|270002550615,6259,270002550615,30,7,ESTADUAL|TO,30133,ALCIDINO VIANA PEREIRA,ALCIDINO VIANA PEREIRA,NaN,...,PARTIDO NOVO,TO,TO,TOCANTINS,2026,1,Eleições Gerais Estaduais 2026,2026-10-04,50–59,4
20528,6259|270002552219,6259,270002552219,27,7,ESTADUAL|TO,27777,CINTHIA CRISTINA MOREIRA SIMPLICIO,CYNTHIA SOBERANO,NaN,...,DEMOCRACIA CRISTÃ,TO,TO,TOCANTINS,2026,1,Eleições Gerais Estaduais 2026,2026-10-04,40–49,3


In [71]:
result

array([5, 5, 4, ..., 4, 3, 4], shape=(20530,))